# Prueba de Concepto: Wavelet Denoising (DWT)

Este notebook aplica en vivo la eliminación de ruido por umbral suave de Wavelet a una señal I/Q del dataset NoisyUAV con un **SNR crítico (-10 dB)**.

In [ ]:
import sys
import os
sys.path.insert(0, os.path.abspath('..'))

import torch
import numpy as np
import matplotlib.pyplot as plt
from NoisyUAV.funciones import TARGET_NOISE, DATA_DIR
from NoisyUAV.funciones.cargador import obtener_una_muestra
from NoisyUAV.funciones.visualizacion import _eje_tiempo, _fmt_ms
from NoisyUAV.funciones.denoising import aplicar_denoising_iq

import warnings
warnings.filterwarnings('ignore')

## 1. Cargar Señal de Dron a SNR Negativo (-10 dB)

In [ ]:
# Vamos a buscar un dron Futaba (Target 0) a SNR de -10 dB
target_drone = 0 
snr_prueba = -20

iq, sid, tgt, snr = obtener_una_muestra(target=target_drone, snr=snr_prueba)

print(f"Cargada muestra ID: {sid} | Target: {tgt} | SNR: {snr} dB")

## 2. Aplicar Transformada Wavelet (DWT)
Aplicaremos el filtro *Daubechies 4 ('db4')* de nivel 2. Esta es la función `denoise_wavelet_1d`.

In [ ]:
# Obtener Arrays crudos para graficar luego
I_raw = iq[0].numpy()
Q_raw = iq[1].numpy()

print("Aplicando PyWavelets...")
I_clean, Q_clean = aplicar_denoising_iq(iq, wavelet='db4', level=2)
print("Denoising completado.")

## 3. Comparación Visual: I/Q Crudo vs I/Q Limpio

In [ ]:
t_sec = _eje_tiempo(len(I_raw))

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(14, 8), sharex=True, sharey=True)

# Panel ORIGINAL (-10 dB)
ax1.plot(t_sec, I_raw, color='salmon', alpha=0.7, label='Original I')
ax1.plot(t_sec, Q_raw, color='lightblue', alpha=0.7, label='Original Q')
ax1.set_title(f'Señal Original (SNR: {snr} dB) - El ruido térmico domina', fontweight='bold')
ax1.legend(loc='upper right')
ax1.grid(True, alpha=0.3)

# Panel DENOISED
ax2.plot(t_sec, I_clean, color='red', alpha=0.9, label='Denoised I')
ax2.plot(t_sec, Q_clean, color='blue', alpha=0.9, label='Denoised Q')
ax2.set_title('Tras DWT (Wavelet Denoising Soft-Threshold) - Recuperando la portadora', fontweight='bold')
ax2.set_xlabel('Tiempo (ms)')
ax2.xaxis.set_major_formatter(plt.FuncFormatter(_fmt_ms))
ax2.legend(loc='upper right')
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 4. Comparación de la Envolvente de Energía (|I+jQ|)
La verdadera prueba es si el **Denoising** logra destapar las ráfagas (bursts) que estaban completamente ocultas en la envolvente.

In [ ]:
env_raw = np.sqrt(I_raw**2 + Q_raw**2)
env_clean = np.sqrt(I_clean**2 + Q_clean**2)

fig, ax = plt.subplots(figsize=(14, 5))
ax.plot(t_sec, env_raw, color='gray', alpha=0.5, label='Envolvente RUIDOSA')
ax.plot(t_sec, env_clean, color='magenta', alpha=0.9, label='Envolvente LIMPIA (Wavelet)')

ax.set_title('Recuperación de Ráfagas (Bursts) mediante Denoising', fontweight='bold')
ax.set_ylabel('Magnitud')
ax.set_xlabel('Tiempo (ms)')
ax.xaxis.set_major_formatter(plt.FuncFormatter(_fmt_ms))
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 5. Comparación en Frecuencia: Espectrogramas (STFT)
Veamos si las matemáticas Wavelet han logrado rescatar los pulsos de radio de la portadora en el dominio Tiempo-Frecuencia.

In [ ]:
from NoisyUAV.funciones import FREQ_MUESTREO
from NoisyUAV.funciones.visualizacion import _fmt_mhz

# Construir señal de valor complejo (I + jQ) para el espectrograma
cplx_raw = I_raw + 1j * Q_raw
cplx_clean = I_clean + 1j * Q_clean

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6), sharex=True, sharey=True)

# Params del STFT
NFFT = 256
overlap = 200

# Espectrograma Ruidoso
Pxx1, freqs1, bins1, im1 = ax1.specgram(
    cplx_raw, NFFT=NFFT, Fs=FREQ_MUESTREO, Fc=0, noverlap=overlap, cmap='viridis'
)
ax1.set_title('Espectrograma: Señal ORIGINAL (-10 dB)', fontweight='bold')
ax1.set_xlabel('Tiempo (ms)')
ax1.set_ylabel('Frecuencia (MHz)')
ax1.xaxis.set_major_formatter(plt.FuncFormatter(lambda x, pos: f"{x*1000:.1f}")) # Sec to ms
ax1.yaxis.set_major_formatter(plt.FuncFormatter(_fmt_mhz))

# Espectrograma Limpio
Pxx2, freqs2, bins2, im2 = ax2.specgram(
    cplx_clean, NFFT=NFFT, Fs=FREQ_MUESTREO, Fc=0, noverlap=overlap, cmap='viridis'
)
ax2.set_title('Espectrograma: Señal WAVELET DENOISED', fontweight='bold', color='darkred')
ax2.set_xlabel('Tiempo (ms)')

fig.colorbar(im1, ax=ax1, label='PSDs (dB)')
fig.colorbar(im2, ax=ax2, label='PSDs (dB)')

plt.tight_layout()
plt.show()